# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIRˆ2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

In [ ]:
# List available record sets and their field/column @ids
record_sets = dataset.metadata.record_sets
print("Available Record Sets (by @id):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    # List fields in record set
    if 'field' in rs:
        print("  Fields (by @id):")
        for f in rs['field']:
            field_id = f['@id'] if isinstance(f, dict) else f
            print(f"    - {field_id}")
    # List columns in record set
    if 'column' in rs:
        print("  Columns (by @id):")
        for c in rs['column']:
            col_id = c['@id'] if isinstance(c, dict) else c
            print(f"    - {col_id}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, extract all record sets into dataframes using their @ids
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    # Load records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

# List loaded DataFrames (by record set @id)
print(f"Loaded DataFrames (by RecordSet @id):\n{list(dataframes.keys())}\n")

# Display fields of the first available DataFrame for further analysis
selected_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if selected_record_set_id and selected_record_set_id in dataframes:
    print(f"Fields/columns in RecordSet {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Example EDA on a numeric field
# First, automatically detect a numeric column in the selected DataFrame
import numpy as np

df = dataframes[selected_record_set_id]
numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a possible categorical field (if available)
    candidate_cats = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field = candidate_cats[0] if candidate_cats else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"Mean {numeric_field_id} grouped by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization: Histogram and Boxplot of the first numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.xlabel(numeric_field_id)

    plt.tight_layout()
    plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIRˆ2 dataset on mass spectrometry analysis of extracellular vesicles using the `mlcroissant` library. We inspected record sets, examined field and column `@id`s, extracted tabular data, performed basic data filtering and normalization, and visualized data distributions. For more advanced analysis, refer to the dataset documentation and the [mlcroissant documentation](https://mlcommons.org/croissant/).